In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [4]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")

!pip install -q transformers peft trl bitsandbytes datasets wandb huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 11.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.0 MB/s eta 0:00:00:00:0100:01


In [ ]:
import transformers
import peft
import trl
import bitsandbytes
import datasets
import wandb

print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("All good! ")

In [ ]:
import wandb
from huggingface_hub import login

wandb.login()
login(token=os.environ["HF_TOKEN"])
print("Logged into both! ")

In [ ]:
from datasets import load_dataset

dataset = load_dataset("medmcqa")
print(dataset)

In [ ]:
sample = dataset["train"][0]
for key, value in sample.items():
    print(f"{key}: {value}")

In [ ]:
print("Question:", dataset["train"][0]["question"])
print("A:", dataset["train"][0]["opa"])
print("B:", dataset["train"][0]["opb"])
print("C:", dataset["train"][0]["opc"])
print("D:", dataset["train"][0]["opd"])

correct = ["A", "B", "C", "D"][dataset["train"][0]["cop"]]
print("Correct answer:", correct)

In [ ]:
import pandas as pd

df_train = pd.DataFrame(dataset["train"])
print("Shape:", df_train.shape)
print("\nMissing values:")
print(df_train.isnull().sum())

In [ ]:
import matplotlib.pyplot as plt

subject_counts = df_train["subject_name"].value_counts()
print("Number of unique subjects:", len(subject_counts))
print("\nTop 10 subjects:")
print(subject_counts.head(10))

In [ ]:
plt.figure(figsize=(12, 5))
subject_counts.head(15).plot(kind="bar", color="steelblue")
plt.title("Top 15 Medical Subjects in MedMCQA Training Set")
plt.xlabel("Subject")
plt.ylabel("Number of Questions")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("subject_distribution.png", dpi=150)
plt.show()
print("Plot saved!")

In [ ]:
def format_prompt(sample):
    options = ["A", "B", "C", "D"]
    correct_letter = options[sample["cop"]]
    
    prompt = f"""You are a medical expert. Answer the following multiple choice question.

Question: {sample["question"]}
A) {sample["opa"]}
B) {sample["opb"]}
C) {sample["opc"]}
D) {sample["opd"]}

Answer: {correct_letter}"""
    return {"text": prompt}

dataset_formatted = dataset.map(format_prompt)
print("Sample formatted prompt:")
print(dataset_formatted["train"][0]["text"])

In [ ]:
train_data = dataset_formatted["train"].shuffle(seed=42).select(range(8000))
val_data = dataset_formatted["validation"].shuffle(seed=42).select(range(500))

print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print("Dataset ready! ")

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-v0.1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_name, token=os.environ["HF_TOKEN"])
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    token=os.environ["HF_TOKEN"]
)

print("Model loaded!")
print(f"Memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


KeyError: 'HF_TOKEN'

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_disable()

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./mistral-medical",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=50,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    report_to="wandb",
    run_name="mistral-medmcqa-qlora",
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=val_data,
    formatting_func=lambda x: x["text"],
    processing_class=tokenizer,
    args=training_args,
)

print("Trainer ready! ")

In [ ]:
trainer.train()
trainer.save_model("./mistral-medical-final")
print("Training complete! ")

In [ ]:
trainer.model.push_to_hub("Asad854/mistral-medical-qlora")
tokenizer.push_to_hub("Asad854/mistral-medical-qlora")
print("Model uploaded to HuggingFace")

In [ ]:
from transformers import pipeline
import torch

pipe = pipeline("text-generation",
               model=trainer.model,
               tokenizer=tokenizer,
               dtype=torch.bfloat16,
               device_map="auto")

correct = 0
total = 300
for i in range(total):
    sample = dataset_formatted["validation"][i]
    question_only = sample["text"].rsplit("Answer:",1)[0] + "Answer:"
    output = pipe(question_only, max_new_tokens=5, do_sample=False)
    generated = output[0]["generated_text"][len(question_only):].strip()
    
    correct_answer = ["A", "B", "C", "D"][dataset["validation"][i]["cop"]]
    
    if generated.startswith(correct_answer):
        correct += 1

finetuned_accuracy = correct / total * 100
print(f"Fine-tuned model accuracy: {finetuned_accuracy:.1f}%")

In [ ]:
import gc
import torch

del trainer
del model
torch.cuda.empty_cache()
gc.collect()
print(f"Memory free: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-v0.1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_name, token=os.environ["HF_TOKEN"])
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    token=os.environ["HF_TOKEN"]
)

print("Base model loaded! ")
print(f"Memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
from transformers import pipeline
import torch

base_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    dtype=torch.bfloat16,
    device_map="auto"
)

correct = 0
total = 300

for i in range(total):
    sample = dataset_formatted["validation"][i]
    question_only = sample["text"].rsplit("Answer:", 1)[0] + "Answer:"
    
    output = base_pipe(question_only, max_new_tokens=5, do_sample=False)
    generated = output[0]["generated_text"][len(question_only):].strip()
    
    correct_answer = ["A", "B", "C", "D"][dataset["validation"][i]["cop"]]
    
    if generated.startswith(correct_answer):
        correct += 1

base_accuracy = correct / total * 100
print(f"Base model accuracy: {base_accuracy:.1f}%")
print(f"Fine-tuned accuracy: {finetuned_accuracy:.1f}%")
print(f"Improvement: +{finetuned_accuracy - base_accuracy:.1f}%")